# Create Work Review Links (oxjob #2481)

One row per candidate review link: `origin_work_id` (the review), `target_work_id` (the work it reviews), where it came
from, and `served` (it passed every rule and is the one row for its edge). `notebooks/end2end/CreateWorkLinks` unions
the served rows into `openalex.works.work_links` as link type `review`, and `/links` serves them.

Sources: Crossref `is-review-of` / `has-review` declarations (forward and inverse collapse into one edge; DOI and PMID
ends resolve; other id types are kept as drops), plus the book-review matches in `openalex.works.book_review_pairs`
(oxjob #2506, a snapshot that job refreshes in place). Rules: both ends resolved through merges and live; the subtype
is judged on the reviewing record (author replies and manuscripts out); a review is not its own work; a review sharing
a (non-software-agent) author with the work it reviews is dropped. Step 5 refuses a build serving 20% fewer links.

Runs as the first task of the Work Links job, after End 2 End. Revert: drop the `review` leg in CreateWorkLinks and
this task from `jobs/work_links.yaml`; the tables can be dropped.

In [0]:
-- 1. Every declared pair, ids normalized. DOI-less ends are kept: a PMID resolves like a DOI; a doi.org URI is a DOI.
CREATE OR REPLACE TABLE openalex.works.work_review_links_declared AS
WITH cr AS (
  SELECT native_id AS doi, relation.`is-review-of` AS iro, relation.`has-review` AS hr
  FROM openalex.crossref.crossref_deduplicated
  WHERE size(relation.`is-review-of`) > 0 OR size(relation.`has-review`) > 0
),
raw AS (
  SELECT 'doi' AS review_ns, doi AS review_raw, coalesce(r.`id-type`, 'doi') AS reviewed_ns, r.id AS reviewed_raw,
         'is-review-of' AS decl
  FROM cr LATERAL VIEW explode(iro) t AS r
  UNION ALL
  SELECT coalesce(r.`id-type`, 'doi'), r.id, 'doi', doi, 'has-review'
  FROM cr LATERAL VIEW explode(hr) t AS r
),
ns AS (
  SELECT decl, review_raw, reviewed_raw,
    CASE WHEN lower(review_ns) = 'uri' AND lower(review_raw) RLIKE 'doi[.]org/10[.]' THEN 'doi' ELSE lower(review_ns) END
      AS review_ns,
    CASE WHEN lower(reviewed_ns) = 'uri' AND lower(reviewed_raw) RLIKE 'doi[.]org/10[.]' THEN 'doi' ELSE lower(reviewed_ns) END
      AS reviewed_ns
  FROM raw
),
ids AS (
  SELECT decl, review_ns, reviewed_ns,
    CASE review_ns
      WHEN 'doi' THEN regexp_replace(lower(trim(review_raw)), '^(https?://(dx[.])?doi[.]org/|doi:)', '')
      WHEN 'pmid' THEN regexp_extract(trim(review_raw), '([0-9]+)', 1)
      ELSE lower(trim(review_raw)) END AS review_id,
    CASE reviewed_ns
      WHEN 'doi' THEN regexp_replace(lower(trim(reviewed_raw)), '^(https?://(dx[.])?doi[.]org/|doi:)', '')
      WHEN 'pmid' THEN regexp_extract(trim(reviewed_raw), '([0-9]+)', 1)
      ELSE lower(trim(reviewed_raw)) END AS reviewed_id
  FROM ns
)
SELECT review_ns, review_id, reviewed_ns, reviewed_id,
       CASE WHEN max(decl = 'is-review-of') AND max(decl = 'has-review') THEN 'both'
            WHEN max(decl = 'is-review-of') THEN 'is-review-of' ELSE 'has-review' END AS declared_by
FROM ids
WHERE review_id <> '' AND reviewed_id <> ''
GROUP BY 1, 2, 3, 4;

In [0]:
-- 2. Resolve both ends to live works (DOI or PMID locations, then merges); judge the subtype on the reviewing record
--    whichever way the pair was declared; apply the rules. First failing rule names the drop.
CREATE OR REPLACE TABLE openalex.works.work_review_links_crossref AS
WITH d AS (SELECT * FROM openalex.works.work_review_links_declared),
ids AS (
  SELECT review_ns AS ns, review_id AS id FROM d WHERE review_ns IN ('doi', 'pmid')
  UNION SELECT reviewed_ns, reviewed_id FROM d WHERE reviewed_ns IN ('doi', 'pmid')
),
id_works AS (
  SELECT l.native_id_namespace AS ns, l.native_id AS id,
         MAX_BY(l.work_id, CASE l.provenance WHEN 'crossref' THEN 3 WHEN 'datacite' THEN 2 ELSE 1 END) AS work_id
  FROM openalex.works.location_work_ids l
  JOIN ids ON l.native_id_namespace = ids.ns AND l.native_id = ids.id
  WHERE l.work_id IS NOT NULL
    AND (l.native_id_namespace = 'pmid' OR l.provenance IN ('crossref', 'datacite'))
  GROUP BY 1, 2
),
resolved AS (
  SELECT i.ns, i.id, coalesce(m.merge_into_id, i.work_id) AS work_id
  FROM id_works i LEFT JOIN openalex.works.merged_work_ids_map m ON m.id = i.work_id
),
live AS (SELECT id FROM openalex.works.openalex_works WHERE id IN (SELECT work_id FROM resolved)),
-- Authors who count for the self-review rule: software-agent seats are left out (two works sharing a "ChatGPT"
-- profile are not the same people; #2505). author.type exists from night 2026-11-22 (#2437); read through JSON so
-- the statement runs before then too.
au AS (
  SELECT work_id,
         array_distinct(transform(
           filter(authorships, a -> a.author.id IS NOT NULL AND a.author.id <> ''
                  AND coalesce(get_json_object(to_json(a.author), '$.type'), '') <> 'software-agent'),
           a -> a.author.id)) AS author_ids
  FROM openalex.works.work_authorships WHERE work_id IN (SELECT work_id FROM resolved)
),
rv AS (
  SELECT native_id AS doi, lower(review.type) AS review_subtype
  FROM openalex.crossref.crossref_deduplicated
  WHERE native_id IN (SELECT review_id FROM d WHERE review_ns = 'doi')
)
SELECT ro.work_id AS origin_work_id, te.work_id AS target_work_id, d.declared_by, rv.review_subtype,
       concat(d.review_ns, ':', d.review_id) AS review_native_id,
       concat(d.reviewed_ns, ':', d.reviewed_id) AS reviewed_native_id,
       CASE
         WHEN d.review_ns NOT IN ('doi', 'pmid') OR d.reviewed_ns NOT IN ('doi', 'pmid') THEN 'id type not resolvable'
         WHEN ro.work_id IS NULL THEN 'review id not in OpenAlex'
         WHEN te.work_id IS NULL THEN 'reviewed id not in OpenAlex'
         WHEN lr.id IS NULL THEN 'review work not live'
         WHEN lt.id IS NULL THEN 'reviewed work not live'
         WHEN rv.review_subtype IN ('author-comment', 'manuscript') THEN concat('subtype ', rv.review_subtype)
         WHEN ro.work_id = te.work_id THEN 'same work'
         WHEN size(array_intersect(coalesce(aur.author_ids, array()), coalesce(aut.author_ids, array()))) > 0
           THEN 'shared author'
       END AS drop_reason
FROM d
LEFT JOIN resolved ro ON ro.ns = d.review_ns AND ro.id = d.review_id
LEFT JOIN resolved te ON te.ns = d.reviewed_ns AND te.id = d.reviewed_id
LEFT JOIN live lr ON lr.id = ro.work_id
LEFT JOIN live lt ON lt.id = te.work_id
LEFT JOIN au aur ON aur.work_id = ro.work_id
LEFT JOIN au aut ON aut.work_id = te.work_id
LEFT JOIN rv ON d.review_ns = 'doi' AND rv.doi = d.review_id;

In [0]:
-- 3. #2506's book-review matches: the same hygiene again (merges since #2506's build, live, self-links, shared
--    authors), since #2506 refreshes its snapshot only now and then.
CREATE OR REPLACE TABLE openalex.works.work_review_links_books AS
WITH b AS (
  SELECT coalesce(m1.merge_into_id, p.review_work_id) AS origin_work_id,
         coalesce(m2.merge_into_id, p.reviewed_work_id) AS target_work_id,
         p.link_rank, p.match_tier
  FROM openalex.works.book_review_pairs p
  LEFT JOIN openalex.works.merged_work_ids_map m1 ON m1.id = p.review_work_id
  LEFT JOIN openalex.works.merged_work_ids_map m2 ON m2.id = p.reviewed_work_id
),
ids AS (SELECT origin_work_id AS work_id FROM b UNION SELECT target_work_id FROM b),
live AS (SELECT id FROM openalex.works.openalex_works WHERE id IN (SELECT work_id FROM ids)),
au AS (
  SELECT work_id,
         array_distinct(transform(
           filter(authorships, a -> a.author.id IS NOT NULL AND a.author.id <> ''
                  AND coalesce(get_json_object(to_json(a.author), '$.type'), '') <> 'software-agent'),
           a -> a.author.id)) AS author_ids
  FROM openalex.works.work_authorships WHERE work_id IN (SELECT work_id FROM ids)
)
SELECT b.origin_work_id, b.target_work_id, b.link_rank, b.match_tier,
       CASE
         WHEN lr.id IS NULL THEN 'review work not live'
         WHEN lt.id IS NULL THEN 'reviewed work not live'
         WHEN b.origin_work_id = b.target_work_id THEN 'same work'
         WHEN size(array_intersect(coalesce(aur.author_ids, array()), coalesce(aut.author_ids, array()))) > 0
           THEN 'shared author'
       END AS drop_reason
FROM b
LEFT JOIN live lr ON lr.id = b.origin_work_id
LEFT JOIN live lt ON lt.id = b.target_work_id
LEFT JOIN au aur ON aur.work_id = b.origin_work_id
LEFT JOIN au aut ON aut.work_id = b.target_work_id;

In [0]:
-- 4. One row per candidate; `served` marks exactly one row per (review, reviewed work) edge that passed every rule
--    (a Crossref declaration first, then a book match). Built beside the served table, checked, then swapped in.
CREATE OR REPLACE TABLE openalex.works.work_review_links_build AS
WITH c AS (
  SELECT origin_work_id, target_work_id, 'crossref' AS link_source, drop_reason, review_subtype, declared_by,
         review_native_id, reviewed_native_id, CAST(NULL AS INT) AS link_rank, CAST(NULL AS STRING) AS match_tier
  FROM openalex.works.work_review_links_crossref
  UNION ALL
  SELECT origin_work_id, target_work_id, 'book-review-match', drop_reason, NULL, NULL, NULL, NULL,
         CAST(link_rank AS INT), match_tier
  FROM openalex.works.work_review_links_books
),
r AS (
  SELECT *, row_number() OVER (
    PARTITION BY origin_work_id, target_work_id
    ORDER BY drop_reason IS NULL DESC, link_source = 'crossref' DESC, review_native_id, reviewed_native_id) AS rn
  FROM c
)
SELECT origin_work_id, target_work_id, link_source,
       (drop_reason IS NULL AND rn = 1 AND origin_work_id IS NOT NULL AND target_work_id IS NOT NULL) AS served,
       CASE WHEN drop_reason IS NULL AND rn > 1 THEN 'duplicate edge' ELSE drop_reason END AS drop_reason,
       review_subtype, declared_by, review_native_id, reviewed_native_id, link_rank, match_tier,
       current_timestamp() AS built_at
FROM r;

In [0]:
-- 5. Fuse: refuse a build that serves 20% fewer links than the one being replaced (a broken input, not real change).
CREATE TABLE IF NOT EXISTS openalex.works.work_review_links LIKE openalex.works.work_review_links_build;

In [0]:
SELECT assert_true(
  (SELECT count_if(served) FROM openalex.works.work_review_links_build)
    >= 0.8 * (SELECT count_if(served) FROM openalex.works.work_review_links),
  'work_review_links_build serves over 20% fewer review links than work_review_links: check the inputs before swapping'
) AS fuse_ok;

In [0]:
-- 6. Swap in.
CREATE OR REPLACE TABLE openalex.works.work_review_links
COMMENT 'oxjob #2481: review links (origin = the review, target = the work reviewed); served rows go to work_links as link type review'
AS SELECT * FROM openalex.works.work_review_links_build;